<a href="https://colab.research.google.com/github/wallynovak/organic_chem/blob/main/chem221_conformational_analysis_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chem 221 — Conformational Analysis: Interactive Notebook

## How to use this notebook

1. **Run Cell 1 (Setup)** first and **wait** for the kernel to restart. A cell is done running when you see a number appear in the brackets, e.g. [1] and a green checkmark below it with the time it took to run that cell.
   After the restart, begin running from **Cell 2** downward —
   do not re-run Cell 1.
2. Run each subsequent cell **in order** using the ▶ button or **Shift + Enter**.
3. Stop at every ✏️ **Worksheet Checkpoint** and answer the listed
   questions on your printed worksheet **before** running the next cell.
   The goal is to **predict first, then verify** — do not skip ahead.
4. Drag the 3D models with your mouse to rotate. Scroll to zoom.

---

## CPK Color Key for 3D Models

| Element | Color in model |
|---|---|
| Carbon | Gray |
| Hydrogen | White |
| Oxygen | Red |
| Nitrogen | Blue |
| **Rotating-bond carbons** | **Gold** |

The two **gold** atoms in each model mark the carbons of the
**C2–C3 bond** being rotated in that section of the notebook.

In [ ]:
#@title Cell 1: Run this cell to install the needed software - Just click the ▶ button

import subprocess, sys

subprocess.check_call(['apt-get', 'install', '-y', 'openbabel'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openbabel', '-q'])

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("ipympl")
install_if_missing("rdkit", "rdkit")
install_if_missing("py3Dmol")
get_ipython().kernel.do_shutdown(restart=True)

print('Programs installed. Kernel shutdown and restarted.\nThe session crashed on purpose.\nDo not rerun this cell. Continue to the next cell.')

In [ ]:
#@title Cell 2: Run this cell to import the needed libraries - Just click the ▶ button

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolTransforms
import py3Dmol

from openbabel import openbabel as ob
from openbabel import pybel

from scipy.signal import savgol_filter

try:
    from google.colab import output, drive
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("All libraries imported successfully. Continue to the next cell.")

In [ ]:
#@title Cell 3: Run this cell to define the conformational energy scanner - Just click the ▶ button

# ── CPK color scheme ──────────────────────────────────────────────────────────

COLORS = {
    'H':  '#FFFFFF',
    'C':  '#909090',
    'N':  '#2944cf',
    'O':  '#d60d0d',
    'F':  '#37c4a9',
    'Cl': '#41bf41',
    'P':  '#FF8000',
    'S':  '#b5b50d',
}
GOLD = '#FFD700'

# ── Molecule builder ──────────────────────────────────────────────────────────

def build_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError('Could not parse SMILES: ' + smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    return mol

# ── Dihedral setter ───────────────────────────────────────────────────────────

def set_dihedral(mol, a, b, c, d, angle, minimize=True):
    mol2 = Chem.Mol(mol)
    conf  = mol2.GetConformer()
    rdMolTransforms.SetDihedralDeg(conf, a, b, c, d, float(angle))
    if minimize:
        props = AllChem.MMFFGetMoleculeProperties(mol2, mmffVariant='MMFF94')
        ff    = AllChem.MMFFGetMoleculeForceField(mol2, props)
        for idx in (a, b, c, d):
            ff.MMFFAddPositionConstraint(idx, 0, 1e4)
        ff.Minimize(maxIts=500)
    return mol2

# ── 3D viewer styler ──────────────────────────────────────────────────────────

def apply_styles(view, mol, highlight_idxs=(), label_map=None, set_camera=True):
    view.removeAllModels()
    view.removeAllLabels()
    view.addModel(Chem.MolToMolBlock(mol), 'mol')
    view.setBackgroundColor('white')

    for elem, col in COLORS.items():
        view.setStyle(
            {'elem': elem},
            {'stick':   {'color': col,  'radius': 0.10},
             'sphere':  {'scale': 0.25, 'color': col}}
        )

    for idx in highlight_idxs:
        view.setStyle(
            {'index': int(idx)},
            {'stick':   {'color': GOLD, 'radius': 0.13},
             'sphere':  {'scale': 0.28, 'color': GOLD}}
        )

    if label_map is None:
        label_map = {}
    conf = mol.GetConformer(0)
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == 'H':
            continue
        idx = atom.GetIdx()
        pos = conf.GetAtomPosition(idx)
        lbl = label_map.get(idx, atom.GetSymbol() + str(idx + 1))
        view.addLabel(lbl, {
            'position':          {'x': float(pos.x),
                                  'y': float(pos.y) + 0.5,
                                  'z': float(pos.z)},
            'fontColor':         'black',
            'backgroundColor':   'white',
            'fontSize':          12,
            'backgroundOpacity': 0.8
        })

    if set_camera:
        view.zoomTo()
        view.zoom(1.8)
    return view

# ── Dihedral energy scanner ───────────────────────────────────────────────────

def scan_dihedral(mol, a, b, c, d, smiles, step=1):
    ob_conv = ob.OBConversion()
    ob_conv.SetInFormat('mol')
    ff_ob = ob.OBForceField.FindForceField('MMFF94')
    if ff_ob is None:
        raise RuntimeError('OpenBabel MMFF94 force field not found.')

    angles = list(range(0, 361, step))
    raw = []

    for angle in angles:
        # Build fresh RDKit geometry and set the target dihedral
        mol_tmp = build_mol(smiles)
        conf = mol_tmp.GetConformer()
        rdMolTransforms.SetDihedralDeg(conf, a, b, c, d, float(angle))

        # Transfer geometry to OpenBabel via mol block
        # This preserves atom ordering so indices a,b,c,d are the same
        mol_block = Chem.MolToMolBlock(mol_tmp)
        ob_mol = ob.OBMol()
        ob_conv.ReadString(ob_mol, mol_block)

        # True torsion constraint — locks dihedral, frees everything else
        # OpenBabel uses 1-based atom indices
        constraints = ob.OBFFConstraints()
        constraints.AddTorsionConstraint(
            a+1, b+1, c+1, d+1, float(angle)
        )

        # Full minimization with torsion locked
        ff = ob.OBForceField.FindForceField('MMFF94')
        ff.Setup(ob_mol, constraints)
        ff.ConjugateGradients(2000)

        raw.append(ff.Energy() * 4.184)

    raw = np.array(raw)
    raw_rel = raw - raw.min()
    smoothed = savgol_filter(raw_rel, window_length=21, polyorder=3)
    smoothed = np.clip(smoothed, 0, None)
    return np.array(angles), smoothed

# ── Conformation explorer (dropdown) ─────────────────────────────────────────

def make_conformation_viewer(mol, a, b, c, d,
                              highlight_idxs=(), label_map=None,
                              key_angles=None):
    if key_angles is None:
        key_angles = {
            0:   'Eclipsed',
            60:  'Staggered',
            120: 'Eclipsed',
            180: 'Staggered',
            240: 'Eclipsed',
            300: 'Staggered'
        }
    if label_map is None:
        label_map = {}

    angle_list = sorted(key_angles.keys())
    conformers = {}
    for ang in angle_list:
        conformers[ang] = set_dihedral(mol, a, b, c, d, ang, minimize=True)

    init_ang = angle_list[0]
    view = py3Dmol.view(width=580, height=400)
    apply_styles(view, conformers[init_ang],
                 highlight_idxs, label_map, set_camera=True)

    viewer_out = widgets.Output()
    with viewer_out:
        display(view)

    dd_options = [
        (str(ang) + '° — ' + key_angles[ang], ang)
        for ang in angle_list
    ]
    dropdown = widgets.Dropdown(
        options=dd_options,
        value=init_ang,
        description='Conformation:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )

    dihedral_label = widgets.HTML(
        value='<b>Current dihedral: ' + str(init_ang) + '°</b>'
    )

    def on_dropdown_change(change):
        ang = change['new']
        apply_styles(view, conformers[ang],
                     highlight_idxs, label_map, set_camera=False)
        view.update()
        dihedral_label.value = (
            '<b>Current dihedral: ' + str(ang) + '°</b>'
        )

    dropdown.observe(on_dropdown_change, names='value')

    header = widgets.HTML(value=(
        '<div style="background:#e8f4f8;padding:8px;'
        'border-radius:5px;margin-bottom:6px;">'
        '<b>3D Conformation Explorer</b> — '
        'Drag the model to rotate; scroll to zoom. '
        '<b>View the molecule looking down the C2-C3 axis.</b> '
        'Gold atoms mark the two carbons of the rotating C2-C3 bond. '
        '<b><u>Explore the different conformations using the dropdown menu.</u></b> '
        '</div>'
    ))

    return widgets.VBox([header, dropdown, dihedral_label, viewer_out])


# ── Energy scanner ────────────────────────────────────────────────────────────

def make_scanner(mol, a, b, c, d,
                 smiles='',
                 highlight_idxs=(), label_map=None,
                 key_angle_labels=None,
                 dihedral_name='C1-C2-C3-C4'):
    if label_map is None:
        label_map = {}

    print('Computing energy profile — please wait...')
    scan_angles, scan_energies = scan_dihedral(mol, a, b, c, d,
                                               smiles=smiles, step=1)
    barrier = float(scan_energies.max())
    min_e   = float(scan_energies.min())
    max_e   = float(scan_energies.max())
    print('Done!  Barrier = ' + str(round(barrier, 1)) + ' kJ/mol.')

    rows = ''
    for ang in sorted(key_angle_labels.keys()):
        idx = int(np.argmin(np.abs(scan_angles - ang)))
        e   = float(scan_energies[idx])
        lbl = key_angle_labels[ang]
        rows += (
            '<tr style="text-align:center;">'
            '<td style="padding:4px 8px;">' + str(ang) + '</td>'
            '<td style="padding:4px 8px;">' + lbl + '</td>'
            '<td style="padding:4px 8px;">' + str(round(e, 2)) + '</td>'
            '</tr>'
        )

    table_html = (
        '<table border="1" style="border-collapse:collapse;'
        'width:94%;margin-top:6px;font-size:13px;">'
        '<tr style="background:#cde8f8;text-align:center;">'
        '<th style="padding:5px;">Dihedral</th>'
        '<th style="padding:5px;">Conformation</th>'
        '<th style="padding:5px;">Relative Energy (kJ/mol)</th>'
        '</tr>'
        + rows +
        '</table>'
    )

    init_angle = 0
    init_idx   = int(np.argmin(np.abs(scan_angles - init_angle)))
    init_e     = float(scan_energies[init_idx])
    mol_init   = set_dihedral(mol, a, b, c, d, init_angle, minimize=False)

    view = py3Dmol.view(width=460, height=370)
    apply_styles(view, mol_init, highlight_idxs, label_map, set_camera=True)

    viewer_out = widgets.Output(
        layout=widgets.Layout(width='46%', min_width='300px')
    )
    with viewer_out:
        display(view)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    fig.patch.set_facecolor('#f8f8f8')
    ax.set_facecolor('#f0f4f8')
    ax.plot(scan_angles, scan_energies, color='steelblue', linewidth=2.5)
    ax.set_xlabel('Dihedral Angle (degrees)', color='#333333', fontsize=10)
    ax.set_ylabel('Relative Energy (kJ/mol)', color='#333333', fontsize=10)
    ax.tick_params(colors='#333333')
    ax.set_xlim(-15, 378)
    ax.set_xticks([0, 60, 120, 180, 240, 300, 360])
    ax.set_ylim(min_e - 1, max_e + 2)
    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')

    dot,  = ax.plot([init_angle], [init_e],
                    'o', color='#FF5722', markersize=9, zorder=5)
    vline = ax.axvline(x=init_angle, color='#FF5722',
                       linewidth=0.8, linestyle='--', alpha=0.6)
    title = ax.set_title(
        'Conformational Energy Profile   |   ' +
        str(round(init_e, 2)) + ' kJ/mol',
        color='#333333', fontsize=11
    )

    ax.annotate('', xy=(350, max_e), xytext=(350, min_e),
                arrowprops=dict(arrowstyle='<->', color='purple', lw=1.2))
    ax.text(354, (min_e + max_e) / 2,
            'DE: ' + str(round(barrier, 1)) + ' kJ/mol',
            color='purple', fontsize=9, ha='left', va='center',
            bbox=dict(facecolor='white', alpha=0.7,
                      edgecolor='none', boxstyle='round,pad=0.2'))
    plt.tight_layout()

    plot_out = widgets.Output(
        layout=widgets.Layout(width='54%', min_width='320px')
    )
    with plot_out:
        plt.show()

    slider = widgets.IntSlider(
        value=init_angle, min=0, max=360, step=1,
        description='Dihedral:',
        continuous_update=True,
        layout=widgets.Layout(width='96%'),
        style={'description_width': '80px'}
    )

    info_label = widgets.HTML(value=(
        '<b>Dihedral (' + dihedral_name + '): '
        + str(init_angle) + ' — '
        + str(round(init_e, 2)) + ' kJ/mol</b>'
    ))

    def on_slider_change(change):
        angle = int(change['new'])
        idx   = int(np.argmin(np.abs(scan_angles - angle)))
        e     = float(scan_energies[idx])

        dot.set_data([angle], [e])
        vline.set_xdata([angle])
        title.set_text(
            'Conformational Energy Profile   |   ' +
            str(round(e, 2)) + ' kJ/mol'
        )
        fig.canvas.draw_idle()

        mol_tmp = set_dihedral(mol, a, b, c, d, angle, minimize=False)
        apply_styles(view, mol_tmp, highlight_idxs, label_map,
                     set_camera=False)
        view.update()

        info_label.value = (
            '<b>Dihedral (' + dihedral_name + '): ' +
            str(angle) + ' — ' +
            str(round(e, 2)) + ' kJ/mol</b>'
        )

    slider.observe(on_slider_change, names='value')

    header = widgets.HTML(value=(
        '<div style="background:#e8f4f8;padding:8px;'
        'border-radius:5px;margin-bottom:4px;">'
        '<b>Energy Scanner</b> — '
        '<b>View the molecule looking down the C2-C3 axis.</b> '
        'Move the slider to rotate the C2-C3 bond. '
        'The orange dot traces the energy curve. '
        'Copy values from the summary table below into your worksheet.'
        '</div>'
    ))

    divider      = widgets.HTML(value='<hr style="margin:8px 0;">')
    table_header = widgets.HTML(
        value='<b>Summary table — record these values on your worksheet</b>'
    )
    table_widget = widgets.HTML(value=table_html)

    return widgets.VBox([
        header,
        info_label,
        slider,
        widgets.HBox([viewer_out, plot_out]),
        divider,
        table_header,
        table_widget
    ])


print('Helper functions defined. Continue to the next cell.')

## ⚖️ Part 1: Butane

In Part 1 you will explore conformations of **butane** (C₄H₁₀)
by rotating around the **C2–C3 bond**.

- **Cell 4** — 3D conformation explorer: dropdown selects six key conformations
- **Cell 5** — interactive energy scanner: slider + energy profile + summary table

---

✏️ **Worksheet Checkpoint — complete before running Cell 4**

> Answer **Q1a** (draw the line-wedge-dash structure of butane, label C1–C4,
> and circle the C2–C3 bond) and **Q1b** (model-kit dihedral-angle questions)
> on your printed worksheet **before** opening the 3D explorer below.

In [ ]:
#@title Cell 4 — Part 1: Butane 3D Conformation Explorer { display-mode: "form" }

butane = build_mol("CCCC")

# Heavy-atom indices after AddHs:
#   C1 = 0,  C2 = 1,  C3 = 2,  C4 = 3
# Dihedral definition:  a=0, b=1, c=2, d=3  (C1-C2-C3-C4)
# Gold highlight: atoms 1 and 2  (C2 and C3, the rotating bond)

butane_labels = {0: "C1", 1: "C2", 2: "C3", 3: "C4"}

butane_view_angles = {
    0:   "Eclipsed — CH₃ aligned with CH₃",
    60:  "Staggered — Gauche",
    120: "Eclipsed — partial",
    180: "Staggered — Anti",
    240: "Eclipsed — partial",
    300: "Staggered — Gauche"
}

display(make_conformation_viewer(
    butane, 0, 1, 2, 3,
    highlight_idxs=(1, 2),
    label_map=butane_labels,
    key_angles=butane_view_angles
))

✏️ **Worksheet Checkpoint — complete before running Cell 5**

> 1. Use the dropdown above to step through all six conformations.
>    Compare each 3D structure to the Newman projections you sketched in **Q1c**.
>    Correct any projections that do not match.
> 2. Complete the **energy estimation table Q1d** using the interaction-energy
>    reference table at the top of your worksheet.
> 3. Sketch your **predicted energy profile Q1e** on the blank axes provided.
>
> **Do not run Cell 5 until Q1c, Q1d, and Q1e are fully complete.**

---

In [ ]:
#@title Cell 5 — Part 1: Butane Energy Scanner { display-mode: "form" }

from google.colab import output
output.enable_custom_widget_manager()

butane_scan_labels = {
    0:   "Eclipsed (CH₃–CH₃ aligned)",
    60:  "Staggered — Gauche",
    120: "Eclipsed",
    180: "Staggered — Anti",
    240: "Eclipsed",
    300: "Staggered — Gauche"
}

get_ipython().run_line_magic('matplotlib', 'widget')
display(make_scanner(
    butane, 0, 1, 2, 3,
    smiles='CCCC',
    highlight_idxs=(1, 2),
    label_map=butane_labels,
    key_angle_labels=butane_scan_labels,
    dihedral_name="C1–C2–C3–C4"
))

✏️ **Worksheet Checkpoint — complete after using Cell 5**

> 1. Move the slider to 0°, 60°, 120°, 180°, 240°, and 300° in turn.
>    Read the computed relative energy at each angle from the **summary table**
>    below the scanner.
> 2. Enter those values in the **Computed** column of comparison table **Q1f**.
> 3. Answer reflection questions **Q1g (i), (ii), and (iii)** on your worksheet.

---

## ⚖️ Part 2: 2-Methylbutane

In Part 2 you will repeat the conformational analysis for
**2-methylbutane** (C₅H₁₂), rotating around the **C2–C3 bond**.

The key structural difference from butane:
**C2 now carries two methyl groups** (the C1 chain-end methyl and an extra
branch methyl), while C3 carries only one methyl and two hydrogens.
This asymmetry changes both the number of distinct staggered conformations
and their relative energies — your job is to predict how before running the scanner.

---

**Atom labels used in the 3D model:**

| Label | Atom |
|---|---|
| C1 | Terminal methyl at the left end of the longest chain |
| C2 | Tertiary carbon — the branch point **(gold)** |
| branch | The extra methyl group bonded to C2 |
| C3 | Next carbon along the chain **(gold)** |
| C4 | Terminal methyl bonded to C3 |

The two **gold** atoms mark C2 and C3, the bond being rotated.

---

✏️ **Worksheet Checkpoint — complete before running Cell 6**

> Answer **Q2a** (draw 2-methylbutane, label all carbons, circle the C2–C3 bond)
> and **Q2b** (model-kit questions: substituents on C2 and C3, and the
> all-anti question) before opening the 3D explorer.

In [ ]:
#@title Cell 6 — Part 2: 2-Methylbutane 3D Conformation Explorer { display-mode: "form" }

methylbutane = build_mol("CC(C)CC")

# SMILES "CC(C)CC" heavy-atom indices after AddHs:
#   0 = C1      (terminal methyl, chain end)
#   1 = C2      (tertiary branch carbon)        <- gold
#   2 = branch  (extra methyl on C2)
#   3 = C3      (next chain carbon)             <- gold
#   4 = C4      (terminal methyl of C3)
#
# Dihedral C1-C2-C3-C4 uses atoms:  a=0, b=1, c=3, d=4
# Rotating bond C2-C3             = atoms 1 and 3
# Note: atom 2 (branch methyl) is bonded to C2 and rotates with it
#       but is NOT one of the four dihedral-definition atoms.

methbu_labels = {
    0: "C1",
    1: "C2",
    2: "branch",
    3: "C3",
    4: "C4"
}

methbu_view_angles = {
    0:   "Eclipsed",
    60:  "Staggered",
    120: "Eclipsed",
    180: "Staggered",
    240: "Eclipsed",
    300: "Staggered"
}

display(make_conformation_viewer(
    methylbutane, 0, 1, 3, 4,
    highlight_idxs=(1, 3),
    label_map=methbu_labels,
    key_angles=methbu_view_angles
))

✏️ **Worksheet Checkpoint — complete before running Cell 7**

> 1. Step through all six conformations using the dropdown.
>    Pay close attention to the **branch** label — it is always bonded to C2
>    but its orientation relative to C3 changes with every 60° rotation.
> 2. Remember: C2 (front carbon in your Newman projections) has
>    **two methyls and one H**; C3 (back carbon) has **one methyl and two H’s**.
>    Compare each 3D view to your Newman projections from **Q2c** and correct
>    any errors.
> 3. Complete the **energy estimation table Q2d**, the **symmetry prediction Q2e**,
>    and the **predicted profile sketch Q2f** on your worksheet.
>
> **Do not run Cell 7 until Q2c, Q2d, Q2e, and Q2f are fully complete.**

---

In [ ]:
#@title Cell 7 — Part 2: 2-Methylbutane Energy Scanner { display-mode: "form" }

methbu_scan_labels = {
    0:   "Eclipsed",
    60:  "Staggered",
    120: "Eclipsed",
    180: "Staggered",
    240: "Eclipsed",
    300: "Staggered"
}

display(make_scanner(
    methylbutane, 0, 1, 3, 4,
    smiles='CC(C)CC',
    highlight_idxs=(1, 3),
    label_map=methbu_labels,
    key_angle_labels=methbu_scan_labels,
    dihedral_name="C1–C2–C3–C4"
))

✏️ **Worksheet Checkpoint — complete after using Cell 7**

> 1. Move the slider to 0°, 60°, 120°, 180°, 240°, and 300°.
>    Read the computed relative energy at each angle from the **summary table**.
> 2. Enter those values in the **Computed** column of comparison table **Q2g**.
> 3. Answer reflection questions **Q2h (i), (ii), and (iii)** on your worksheet.

---

---

## 🏁 Notebook Complete

In this activity you:

- Explored 3D conformations of **butane** and **2-methylbutane** using
  a dropdown selector and an interactive dihedral slider
- Verified your hand-drawn Newman projections against computed 3D geometry
- Compared **additive hand-estimates** to **MMFF94 force-field calculations**
- Discovered that 2-methylbutane’s three staggered conformations are
  **not all equal in energy** because C2 carries two methyl groups —
  one staggered arrangement places both methyls gauche to C4 simultaneously

📌 **Before submitting your worksheet**, confirm that:

- All six Newman projection circles are drawn and labelled for both molecules
- Both comparison tables (Q1f and Q2g) contain computed values from the scanner
- All written reflection questions Q1g and Q2h are answered in complete sentences
- Your two predicted energy sketches show labelled minima and maxima